# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（共同作者不等於已完成個別工作）
- Date first created: 20261127
- Background: 296個原五欄分類鍵之v2研究分類支援表，僅IF產業服務及B第二代展會中心兩鍵三圖徵修訂為mixed_special_use。保留v1原分類、原verification_status、新分類、版本、理由、證據、來源ID與影響面積；不是第二份B2分析資料或法定分區變更。 本文件獨立讀正式CSV／schema，不重跑GIS或舊Notebook，未建立正式Finding。
- Unit of observation: 五欄原始分類複合鍵，296列。
- Sample period: 單一都市計畫固定快照，沿Unit1.4內容標示2025-01、資源2026-04-09及實際DBF日期分開記錄；均不等於全部個案法定生效日，不宣稱2024／2025兩套觀測或事故前環境。
- Known issues: 面層原CRS缺PRJ，僅記憶體EPSG:3826工作假設B級，非A級認證。原有效面不修復，105無效面算法一致候選採linework；38方法敏感以三法面候選聯集保守扣除，buffer0只作比較；非面構件另記。固定原里界不修補，正式分母沿Unit2.2。衍生分區overlay採10^-4m一致運算網格控制GEOS數值殘留，原圖資／原里界座標不變，這不是實際位置精度認證；逐里面積關係容忍0.1m²、衍生分類交集檢查容忍0.01m²；全市聚合面積關係另容忍0.1m²（小於全市面積的4×10^-10），不clip比例。陽明山公園仍unclassified，不按景觀猜公園。五類不正規化，特殊混合等已知非五主類不當未知。不確定來源可重疊，總量用union。已辨識零不代表確定不存在；處理失敗保留ID／NA。L=observed，U=min(1,L+Q/A)，是依工作分類正確及敏感範圍包絡足夠等假設的條件範圍，不是信賴區間、法定真值或土地使用真值界，各上界不能同時實現，不以中點補值。品質門檻僅候選，不排除其他資料集里。城市union、逐里面積和加權、里算術平均及中位數分開；原里界微量重疊保留。D改良版／Prediction暫緩，無正式EDA／模型／推論。成員交付確認與DA／PM驗收待完成。
- Definition for each variable: 下方完整21欄字典逐一列dtype／定義／單位／來源／缺失規則。五欄複合鍵不得簡化成單一代碼；原空字串保留。
- Data path: `data/processed/urban_zoning_crosswalk_v2.csv`
- Source data: 原都市計畫六配套（點層不算面積）、固定里界五配套、正式里主表、v1 crosswalk及既有證據，25項來源／helper固定SHA見B2_INPUTS；12項A／B1／C僅交付鍵檢查，不作分區特徵值輸入。
- Processing code: [處理Notebook](../code/prepare_taipei_dataset_b2.ipynb)；[helper](../code/unit25_zoning_support.py)。Producer SHA `d3666314a709587732dc23905837d3c62014cd3cc1b142c7221037dc21d1b00e`。
- Role: processing_support_zoning_classification
- Verification status: 正式資料296×21、139,463 bytes，先經處理NotebookGIS／分類／面積與精確讀回核對；下方Assertions保存本文件乾淨kernel實際結果。技術通過不代表人工或官方核定。
- SHA-256: `3af29de364a10cd135252b9851ce6bbb74586ab47d5c5d68ade036e4a2960583`
- Schema: `data/processed/urban_zoning_crosswalk_v2_schema.json`；SHA `5925bf3b77fd129bfd1e7eb4aaf765596b4d26fe0228b1c34f3a9935f9711a5f`。
- Downstream analysis input: `data/processed/urban_zoning_crosswalk_v2.csv`（與Data path相同；crosswalk為處理支援而非研究觀察樣本）

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。Deliverables：本資料／schema／獨立說明及共用處理。Actual Contribution：Codex協助，保留共同作者，不推定成員實際交付／驗收完成。

數值語義：核心observed與所有條件式L／U嚴格0–1；十個品質來源比例可有浮點微量超界，容忍為每里0.1m²除以完整里面積。原比例不clip，quality_share_roundoff_flag及quality_share_roundoff_max_excess_m2記錄超界與超額面積；不把非分區邊界當有效分區分類。


## Packages、來源與正式版本

In [1]:
from pathlib import Path
import sys,json,hashlib,platform
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():
        PROJECT_ROOT=p;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for part in iter(lambda:stream.read(4*1024*1024),b''):h.update(part)
    return h.hexdigest()
assert sha(PROJECT_ROOT/'data/code/unit25_zoning_support.py')=='a5d1a8bbc8b0473921340b01af09506902123030502a61035def77c83af7b2a4'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
import unit25_zoning_support as u
B2_INPUTS={'data/raw/細計-面.shp': '67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee', 'data/raw/細計-面.shx': '277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69', 'data/raw/細計-面.dbf': '29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4', 'data/raw/細計-文.shp': 'fa1767fc78c69b7683ec53e0fab8937bac6c33868bedee3c3dcc2ed38dcab731', 'data/raw/細計-文.shx': '32d642964d494f49d08830f9c12857f9f4995b56eebf09ef2be994034b864760', 'data/raw/細計-文.dbf': 'eba92881fc11e574f24024380c946d73cf213e86040663c4070d15b1d552a7be', 'data/raw/G97_A_CAVLGE_P.shp': 'f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc', 'data/raw/G97_A_CAVLGE_P.shx': 'ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e', 'data/raw/G97_A_CAVLGE_P.dbf': '2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577', 'data/raw/G97_A_CAVLGE_P.prj': '8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5', 'data/raw/G97_A_CAVLGE_P.cpg': '3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824', 'data/processed/taipei_village_base.csv': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'data/processed/taipei_village_base_schema.json': 'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070', 'data/processed/taipei_village_base_documentation.ipynb': 'a843d587788b4ea6a943cc05d9413908cf8b66b2f73f9fdbec01e2ab52edd535', 'data/processed/urban_zoning_crosswalk.csv': '4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f', 'data/processed/urban_zoning_crosswalk_documentation.ipynb': 'b931add933d943890b9ec8fbed463836d28838632cec3a5f0636ff0f6b9fa36d', 'data/raw/taipei_urban_planning_raw_documentation.ipynb': 'bb3dd6bca248ce866b30766762d100cd926e7cbed30f0165381f531709b8aa8f', 'data/raw/unit14_urban_planning_evidence.zip': '695e15f7230c076b2b827f72b4337273ff795f6fc7e9c52680641c972dd72730', 'data/raw/unit14_urban_planning_evidence_documentation.ipynb': 'ec2e9b002441d10b26b426f6d43cf6c5dd9c598ca058cda81a2f9f058e02dc55', 'data/raw/unit15_zoning_classification_evidence.zip': '82effe1535fc7577d73accf81ee0663daa78944a11517b9e50c7bc2e197ba9b9', 'data/raw/unit15_zoning_classification_evidence_documentation.ipynb': '15f164fd91f2b6459c56e789001bc928e8dd96f01e14e2ff4c311e758ce14c35', 'data/code/taipei_urban_planning_crs_validation.ipynb': 'cf9cd6a6d46a943fab2f148db114d1ad49a605a664a2ee6109761df8e588b839', 'data/code/taipei_urban_planning_geometry_validation.ipynb': 'b5c8b2358dc9774c744639c94d85961e44ec8230f223d63387cd6846605e04b7', 'data/code/taipei_data_integration_readiness.ipynb': '6048ff52d7653c9ec9d4b32cf2b78da2fb1b22dc0db0164cc48c4d2d98dd338b', 'data/code/unit25_zoning_support.py': 'a5d1a8bbc8b0473921340b01af09506902123030502a61035def77c83af7b2a4'}
HANDOFF_ONLY_INPUTS={'data/processed/taipei_dataset_a_annual.csv': '4e264176c970263777eb4c564f427e1f839a45b31552a37bc4bdbfbed6b09422', 'data/processed/taipei_dataset_a_annual_schema.json': '4d0c5b7b307b1cca4261a9595cd1ca894f549d88e06bed1b0b071dcf7c40fb21', 'data/processed/taipei_dataset_a_pooled.csv': '081c7dd42c94e16a5684f27edaebd310a9c556bae7460783dd87fde8d26ea46a', 'data/processed/taipei_dataset_a_pooled_schema.json': '7878146fac39d4bfabee965b6001d0cf5e101ca1d76778599fec4b480ad59b5a', 'data/processed/taipei_dataset_b1_annual.csv': 'd4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795', 'data/processed/taipei_dataset_b1_annual_schema.json': '89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca', 'data/processed/taipei_dataset_b1_pooled.csv': '3b02fb7b338b59f2552972ccb98efa0d723e525826b301cdda021efa9ddd4e50', 'data/processed/taipei_dataset_b1_pooled_schema.json': '63f46e1ae0cba9c19549ee5f8b9ad39084a22f5d2d738b62567b1b3f5e86f340', 'data/processed/taipei_dataset_c_annual.csv': '9ab1970ed0f790ebdf31e1726094493305da00ac4efaf84b8a554f7debfa560e', 'data/processed/taipei_dataset_c_annual_schema.json': '4a8151a9a5dcd716cc03312b303c4455cb47210192c43b86cf3a0e06d268cb6f', 'data/processed/taipei_dataset_c_pooled.csv': 'caceea61ff906b3e2f0b4796a31af3373a929307a0c7f7094fb9373794f76bc6', 'data/processed/taipei_dataset_c_pooled_schema.json': '44baac19975650d792e23e81c5463b84e00f299c9c606ef0f95abf8b3756d3ab'}
u.verify(PROJECT_ROOT,B2_INPUTS);u.verify(PROJECT_ROOT,HANDOFF_ONLY_INPUTS)
print('Python',platform.python_version(),'B2來源／helper hash',len(B2_INPUTS),'僅交付鍵來源hash',len(HANDOFF_ONLY_INPUTS))

EXPECTED_OUTPUTS={'data/processed/taipei_dataset_b2.csv': '1af7ce133e62a1188e8755a1e4e7bc55db8b89d639f089f1c75319c8a9ee0269', 'data/processed/taipei_dataset_b2_schema.json': 'd5857362444df6bea1f606330d0363d792cc1a7f56812c9bb4917ce1b5e3a102', 'data/processed/urban_zoning_crosswalk_v2.csv': '3af29de364a10cd135252b9851ce6bbb74586ab47d5c5d68ade036e4a2960583', 'data/processed/urban_zoning_crosswalk_v2_schema.json': '5925bf3b77fd129bfd1e7eb4aaf765596b4d26fe0228b1c34f3a9935f9711a5f'}
assert sha(PROJECT_ROOT/'data/code/prepare_taipei_dataset_b2.ipynb')=='d3666314a709587732dc23905837d3c62014cd3cc1b142c7221037dc21d1b00e'
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h,('OUTPUT HASH MISMATCH',rel)
print('四正式輸出與producer版本核對通過；只讀saved files')


Python 3.12.2 B2來源／helper hash 25 僅交付鍵來源hash 12
四正式輸出與producer版本核對通過；只讀saved files


## Change for Your Own Data

精確路徑與hash先核對，再按schema讀取，保留原空字串、nullable dtype及`__MISSING__`，不使用預設NA推論或猜版本。

In [2]:
saved={};contracts={}
for name in [u.B2,u.CW]:
    contract=json.loads((PROJECT_ROOT/f'data/processed/{name}_schema.json').read_text(encoding='utf-8'))
    assert contract['source_inputs']==B2_INPUTS and contract['handoff_only_inputs']==HANDOFF_ONLY_INPUTS
    saved[name]=u.read_csv(PROJECT_ROOT/f'data/processed/{name}.csv',contract);contracts[name]=contract
base_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_village_base_schema.json').read_text(encoding='utf-8'))
base=u.read_csv(PROJECT_ROOT/'data/processed/taipei_village_base.csv',base_contract)
old=pd.read_csv(PROJECT_ROOT/'data/processed/urban_zoning_crosswalk.csv',encoding='utf-8-sig',dtype='string',keep_default_na=False)
u.validate_b2(saved[u.B2],base);u.validate_crosswalk(saved[u.CW],old)
for col in old:
    if col=='research_zoning_category':continue
    pd.testing.assert_series_equal(saved[u.CW][col].astype('string'),old[col].astype('string'),check_names=False,check_exact=True)
STEM='urban_zoning_crosswalk_v2'
frame=saved[STEM];contract=contracts[STEM]
assert frame.shape==(296,21)
print('Rows',len(frame),'Columns',frame.shape[1],'primary key',contract['primary_key'])

Rows 296 Columns 21 primary key ['source_layer_code', 'original_zoning_code', 'original_zoning_name', 'original_zoning_description', 'original_zoning_abbreviation']


## Definition for each variable

In [3]:
dictionary=pd.DataFrame(contract['columns'])
u.show('Definition for each variable：完整資料字典',dictionary,block=16)
assert list(frame)==dictionary['name'].tolist()
assert [str(frame[c].dtype) for c in frame]==dictionary['dtype'].tolist()

**Definition for each variable：完整資料字典**

,name,dtype,definition,unit,source,missing_rule
0,source_layer_code,string,v1原欄位保留：source_layer_code,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
1,original_zoning_code,string,v1原欄位保留：original_zoning_code,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
2,original_zoning_name,string,v1原欄位保留：original_zoning_name,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
3,original_zoning_description,string,v1原欄位保留：original_zoning_description,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
4,original_zoning_abbreviation,string,v1原欄位保留：original_zoning_abbreviation,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
5,research_zoning_category,string,v2研究工作分類；僅指定2鍵3筆由industrial／commercial改mixed_special_use,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
6,eligibility_status,string,v1原欄位保留：eligibility_status,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
7,classification_method,string,v1原欄位保留：classification_method,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
8,verification_status,string,原official_supported／research_classification／ambiguous／excluded狀態保留，不提升為官方認證,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
9,source_evidence,string,v1原欄位保留：source_evidence,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA


,name,dtype,definition,unit,source,missing_rule
16,revision_evidence_status,string,本次研究採用或原證據維持；與原verification_status分開,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
17,classification_version,string,v2_unit25，保留v1原檔,not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
18,affected_record_ids_json,string,受指定修訂的來源穩定ID JSON陣列，未修訂為[],not applicable,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
19,revision_affected_record_count,Int64,此鍵指定修訂圖徵數；未修訂為0,source feature count,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA
20,revision_reference_union_area_m2,Float64,受修訂圖徵linework參考聯集面積；非真實／法定面積認證,m²,v1 five-field compound key / Unit1.6 / Unit2.5 adopted research revision,原空字串保留；未修訂原因空字串、IDs為[]；__MISSING__僅真正NA


## Summary

### Sample

In [4]:
u.show('實際前10列主要來源／地理欄',frame.iloc[:10,:18])
u.show('前10列原分類／新分類與修訂證據',frame.iloc[:10][['original_zoning_code','original_research_zoning_category','research_zoning_category','verification_status','classification_revision_applied','revision_evidence_status']])

**實際前10列主要來源／地理欄**

,source_layer_code,original_zoning_code,original_zoning_name,original_zoning_description,original_zoning_abbreviation,research_zoning_category,eligibility_status,classification_method,verification_status,source_evidence,original_record_count,notes,original_research_zoning_category,classification_revision_applied,classification_revision_reason,revision_evidence,revision_evidence_status,classification_version
0,1,R1,第一種住宅區,,住1,residential,eligible_zoning,explicit_reviewed_compound_key_v1,official_supported,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,107,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。,residential,False,,,unchanged_original_evidence,v2_unit25
1,1,R1A,第一種住宅區(特),,住1(特),residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,5,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。,residential,False,,,unchanged_original_evidence,v2_unit25
2,10,R,住宅區,,住,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,156,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。,residential,False,,,unchanged_original_evidence,v2_unit25
3,10,R5,特定休閒旅館住宅專用區,,休閒,mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,25,特定專用、混合或聯合開發等獨立類；即使名稱含住商工，不直接併入五主類，個案細則未逐案認證。,mixed_special_use,False,,,unchanged_original_evidence,v2_unit25
4,10,RA,第壹類住宅用地,,住1類,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,3,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。,residential,False,,,unchanged_original_evidence,v2_unit25
5,10,RAA,住宅區(特),,住(特),residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,8,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。,residential,False,,,unchanged_original_evidence,v2_unit25
6,10,RB,第貳類住宅用地,,住2類,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,14,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。,residential,False,,,unchanged_original_evidence,v2_unit25
7,10,RC,第參類住宅用地,,住3類,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,18,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。,residential,False,,,unchanged_original_evidence,v2_unit25
8,10,RD,駐華使領館辦公暨住宅專用區,,使館住專,mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,1,特定專用、混合或聯合開發等獨立類；即使名稱含住商工，不直接併入五主類，個案細則未逐案認證。,mixed_special_use,False,,,unchanged_original_evidence,v2_unit25
9,10,REA,特定專用區(三)(供住宅使用),,特(三),mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,19,特定專用、混合或聯合開發等獨立類；即使名稱含住商工，不直接併入五主類，個案細則未逐案認證。,mixed_special_use,False,,,unchanged_original_evidence,v2_unit25


**前10列原分類／新分類與修訂證據**

,original_zoning_code,original_research_zoning_category,research_zoning_category,verification_status,classification_revision_applied,revision_evidence_status
0,R1,residential,residential,official_supported,False,unchanged_original_evidence
1,R1A,residential,residential,research_classification,False,unchanged_original_evidence
2,R,residential,residential,research_classification,False,unchanged_original_evidence
3,R5,mixed_special_use,mixed_special_use,research_classification,False,unchanged_original_evidence
4,RA,residential,residential,research_classification,False,unchanged_original_evidence
5,RAA,residential,residential,research_classification,False,unchanged_original_evidence
6,RB,residential,residential,research_classification,False,unchanged_original_evidence
7,RC,residential,residential,research_classification,False,unchanged_original_evidence
8,RD,mixed_special_use,mixed_special_use,research_classification,False,unchanged_original_evidence
9,REA,mixed_special_use,mixed_special_use,research_classification,False,unchanged_original_evidence


### Summary Stats

In [5]:
u.show('v1／v2分類鍵數與圖徵數',frame.groupby(['original_research_zoning_category','research_zoning_category']).agg(compound_keys=('original_record_count','size'),source_features=('original_record_count','sum')).reset_index())
u.show('原證據狀態保留',frame.groupby('verification_status').agg(keys=('original_record_count','size'),features=('original_record_count','sum')).reset_index())
u.show('精確兩修訂鍵與全部追溯資訊',frame.loc[frame.classification_revision_applied])

**v1／v2分類鍵數與圖徵數**

,original_research_zoning_category,research_zoning_category,compound_keys,source_features
0,agriculture_conservation,agriculture_conservation,2,285
1,commercial,commercial,22,2903
2,commercial,mixed_special_use,1,1
3,educational,educational,35,425
4,industrial,industrial,15,215
5,industrial,mixed_special_use,1,2
6,mixed_special_use,mixed_special_use,55,540
7,non_zoning_boundary,non_zoning_boundary,2,3
8,other_classified,other_classified,12,95
9,other_public_facilities,other_public_facilities,85,1134


**原證據狀態保留**

,verification_status,keys,features
0,ambiguous,1,1
1,excluded_non_zoning,2,3
2,official_supported,18,8468
3,research_classification,275,7046


**精確兩修訂鍵與全部追溯資訊**

,source_layer_code,original_zoning_code,original_zoning_name,original_zoning_description,original_zoning_abbreviation,research_zoning_category,eligibility_status,classification_method,verification_status,source_evidence,original_record_count,notes,original_research_zoning_category,classification_revision_applied,classification_revision_reason,revision_evidence,revision_evidence_status,classification_version,affected_record_ids_json,revision_affected_record_count,revision_reference_union_area_m2
27,11,B,商業區,(供第二代展會中心使用),商,mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,1,明列商業、業務或辦公服務之研究寬類；特殊條件未作地塊法定認證。B含展會中心指定用途，另列保守排除方案待審。,commercial,True,B明示第二代展會中心用途，沿Unit1.6指定研究修訂,data/code/taipei_data_integration_readiness.ipynb#section-5; Unit2.5 explicit instruction; original five-field compound key,research_revision_adopted_unit25_not_legal_change,v2_unit25,"[""67d2ecc46938:12205""]",1,33594.480341
85,25,IF,產業服務區,,產服,mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,2,工業／科技工業及產業服務之研究寬類；IF產業服務為研究歸類，是否納主要工業需學生審查。,industrial,True,IF產業服務不直接等同工業功能，沿Unit1.6指定研究修訂,data/code/taipei_data_integration_readiness.ipynb#section-5; Unit2.5 explicit instruction; original five-field compound key,research_revision_adopted_unit25_not_legal_change,v2_unit25,"[""67d2ecc46938:10051"", ""67d2ecc46938:13401""]",2,39016.528867


### 品質、修訂與解釋限制

In [6]:
assert len(frame.loc[~frame.classification_revision_applied])==294
for _,row in frame.iterrows():
    ids=json.loads(row.affected_record_ids_json)
    assert len(ids)==int(row.revision_affected_record_count)
    assert all(x.startswith(B2_INPUTS['data/raw/細計-面.shp'][:12]+':') for x in ids)
u.show('v1與v2差異限定兩完整複合鍵',frame.loc[frame.classification_revision_applied,u.OUTKEY+['original_research_zoning_category','research_zoning_category','revision_affected_record_count','revision_reference_union_area_m2','classification_revision_reason','revision_evidence_status']])
print('舊crosswalk與原verification_status保留；其餘294鍵原欄位均相同；mixed_special_use仍為已知工作分類')

**v1與v2差異限定兩完整複合鍵**

,source_layer_code,original_zoning_code,original_zoning_name,original_zoning_description,original_zoning_abbreviation,original_research_zoning_category,research_zoning_category,revision_affected_record_count,revision_reference_union_area_m2,classification_revision_reason,revision_evidence_status
27,11,B,商業區,(供第二代展會中心使用),商,commercial,mixed_special_use,1,33594.480341,B明示第二代展會中心用途，沿Unit1.6指定研究修訂,research_revision_adopted_unit25_not_legal_change
85,25,IF,產業服務區,,產服,industrial,mixed_special_use,2,39016.528867,IF產業服務不直接等同工業功能，沿Unit1.6指定研究修訂,research_revision_adopted_unit25_not_legal_change


舊crosswalk與原verification_status保留；其餘294鍵原欄位均相同；mixed_special_use仍為已知工作分類


### 使用範例與技術交付

In [7]:
key=frame[u.OUTKEY].copy()
joined=key.merge(frame,on=u.OUTKEY,how='left',validate='one_to_one')
assert len(joined)==296
print('原圖徵須以五欄鍵many-to-one連本表，不能只用分區代碼或刪相同幾何來源；此文件不重讀GIS')

原圖徵須以五欄鍵many-to-one連本表，不能只用分區代碼或刪相同幾何來源；此文件不重讀GIS


### Assertions

獨立saved-file驗證不重跑GIS；技術通過仍待人工驗收。

In [8]:
u.validate_b2(saved[u.B2],base);u.validate_crosswalk(saved[u.CW],old)
u.verify(PROJECT_ROOT,B2_INPUTS);u.verify(PROJECT_ROOT,HANDOFF_ONLY_INPUTS)
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h
again=u.read_csv(PROJECT_ROOT/'data/processed/urban_zoning_crosswalk_v2.csv',contract)
pd.testing.assert_frame_equal(again,frame,check_exact=True)
result={'status':'passed','dataset':STEM,'shape':list(frame.shape),'sha256':sha(PROJECT_ROOT/'data/processed/urban_zoning_crosswalk_v2.csv'),'schema_sha256':sha(PROJECT_ROOT/'data/processed/urban_zoning_crosswalk_v2_schema.json'),'independent_saved_file_read':True,'human_acceptance':'DE / DA / PM pending'}
temp=PROJECT_ROOT/'data/temp/unit25_validation';temp.mkdir(parents=True,exist_ok=True)
(temp/(STEM+'_documentation_summary.json')).write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))

{
  "status": "passed",
  "dataset": "urban_zoning_crosswalk_v2",
  "shape": [
    296,
    21
  ],
  "sha256": "3af29de364a10cd135252b9851ce6bbb74586ab47d5c5d68ade036e4a2960583",
  "schema_sha256": "5925bf3b77fd129bfd1e7eb4aaf765596b4d26fe0228b1c34f3a9935f9711a5f",
  "independent_saved_file_read": true,
  "human_acceptance": "DE / DA / PM pending"
}
